# 10 - SHACL Shape Parser (F6.1)

Parses SHACL shapes to extract validation constraints for pre-load data quality checks.

**Input**: `bronze_triples` Delta table (containing SHACL shape definitions)
**Output**: `silver_shacl_shapes` table + JSON summary for UI

## SHACL Constructs Parsed

| Construct | Description |
|-----------|-------------|
| sh:NodeShape | Shape targeting a class |
| sh:PropertyShape | Constraints on a property |
| sh:targetClass | Explicit target class |
| sh:path | Property path for constraint |
| sh:datatype | Expected XSD datatype |
| sh:minCount / sh:maxCount | Cardinality constraints |
| sh:class | Expected class for object values |
| sh:nodeKind | IRI, Literal, BlankNode, etc. |
| sh:pattern | Regex pattern for values |
| sh:minInclusive / sh:maxInclusive | Value range |
| sh:in | Enumerated allowed values |

## Implicit Class Targeting (SHACL 2.1.3)

When a NodeShape is also declared as `owl:Class` or `rdfs:Class`, it implicitly targets itself.
This is common in ontologies like NEN 2660-2 where class and shape share the same URI.

In [ ]:
# Configuration
INPUT_TABLE = "bronze_triples"
OUTPUT_TABLE = "silver_shacl_shapes"
OUTPUT_JSON = "/lakehouse/default/Files/config/shacl_shapes.json"

In [ ]:
from pyspark.sql import functions as F
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
import json

# SHACL namespace
SH = "http://www.w3.org/ns/shacl#"

# SHACL predicates
SH_NODE_SHAPE = SH + "NodeShape"
SH_PROPERTY_SHAPE = SH + "PropertyShape"
SH_TARGET_CLASS = SH + "targetClass"
SH_TARGET_NODE = SH + "targetNode"
SH_TARGET_SUBJ_OF = SH + "targetSubjectsOf"
SH_TARGET_OBJ_OF = SH + "targetObjectsOf"
SH_PROPERTY = SH + "property"
SH_PATH = SH + "path"
SH_DATATYPE = SH + "datatype"
SH_CLASS = SH + "class"
SH_NODE_KIND = SH + "nodeKind"
SH_MIN_COUNT = SH + "minCount"
SH_MAX_COUNT = SH + "maxCount"
SH_MIN_LENGTH = SH + "minLength"
SH_MAX_LENGTH = SH + "maxLength"
SH_PATTERN = SH + "pattern"
SH_MIN_INCLUSIVE = SH + "minInclusive"
SH_MAX_INCLUSIVE = SH + "maxInclusive"
SH_MIN_EXCLUSIVE = SH + "minExclusive"
SH_MAX_EXCLUSIVE = SH + "maxExclusive"
SH_IN = SH + "in"
SH_NAME = SH + "name"
SH_DESCRIPTION = SH + "description"
SH_MESSAGE = SH + "message"
SH_SEVERITY = SH + "severity"
SH_CLOSED = SH + "closed"
SH_IGNORED_PROPS = SH + "ignoredProperties"

# RDF predicates
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
RDF_FIRST = "http://www.w3.org/1999/02/22-rdf-syntax-ns#first"
RDF_REST = "http://www.w3.org/1999/02/22-rdf-syntax-ns#rest"
RDF_NIL = "http://www.w3.org/1999/02/22-rdf-syntax-ns#nil"

# OWL/RDFS class types (for implicit targeting detection per SHACL 2.1.3)
OWL_CLASS = "http://www.w3.org/2002/07/owl#Class"
RDFS_CLASS = "http://www.w3.org/2000/01/rdf-schema#Class"

In [ ]:
@dataclass
class PropertyConstraint:
    """Constraint from a sh:PropertyShape."""
    path: str = None
    path_name: str = None
    datatype: str = None
    node_kind: str = None
    class_constraint: str = None
    min_count: int = None
    max_count: int = None
    min_length: int = None
    max_length: int = None
    pattern: str = None
    min_inclusive: str = None
    max_inclusive: str = None
    min_exclusive: str = None
    max_exclusive: str = None
    allowed_values: List[str] = field(default_factory=list)
    name: str = None
    description: str = None
    message: str = None
    severity: str = None
    
    def to_dict(self) -> dict:
        d = {}
        if self.path: d["path"] = self.path
        if self.path_name: d["pathName"] = self.path_name
        if self.datatype: d["datatype"] = self.datatype
        if self.node_kind: d["nodeKind"] = self.node_kind
        if self.class_constraint: d["class"] = self.class_constraint
        if self.min_count is not None: d["minCount"] = self.min_count
        if self.max_count is not None: d["maxCount"] = self.max_count
        if self.min_length is not None: d["minLength"] = self.min_length
        if self.max_length is not None: d["maxLength"] = self.max_length
        if self.pattern: d["pattern"] = self.pattern
        if self.min_inclusive: d["minInclusive"] = self.min_inclusive
        if self.max_inclusive: d["maxInclusive"] = self.max_inclusive
        if self.min_exclusive: d["minExclusive"] = self.min_exclusive
        if self.max_exclusive: d["maxExclusive"] = self.max_exclusive
        if self.allowed_values: d["allowedValues"] = self.allowed_values
        if self.name: d["name"] = self.name
        if self.description: d["description"] = self.description
        if self.message: d["message"] = self.message
        if self.severity: d["severity"] = self.severity
        return d


@dataclass
class NodeShape:
    """A sh:NodeShape with its constraints."""
    uri: str
    name: str = None
    target_class: str = None
    target_node: str = None
    is_closed: bool = False
    property_constraints: List[PropertyConstraint] = field(default_factory=list)
    description: str = None
    message: str = None
    severity: str = None
    
    def to_dict(self) -> dict:
        return {
            "uri": self.uri,
            "name": self.name or self.extract_local_name(self.uri),
            "targetClass": self.target_class,
            "targetNode": self.target_node,
            "isClosed": self.is_closed,
            "propertyConstraints": [pc.to_dict() for pc in self.property_constraints],
            "description": self.description,
            "message": self.message,
            "severity": self.severity,
        }
    
    @staticmethod
    def extract_local_name(uri: str) -> str:
        if "#" in uri: return uri.split("#")[-1]
        if "/" in uri: return uri.split("/")[-1]
        return uri

In [ ]:
# Load triples
df_triples = spark.table(INPUT_TABLE)
print(f"Loaded {df_triples.count()} triples from '{INPUT_TABLE}'")

# Check for SHACL content
shacl_count = df_triples.filter(
    F.col("predicate").startswith(SH) |
    (F.col("predicate") == RDF_TYPE) & F.col("object").startswith(SH)
).count()

print(f"SHACL-related triples: {shacl_count}")

if shacl_count == 0:
    print("\nWARNING: No SHACL shapes found in the data.")
    print("Make sure to load nen2660-shacl.ttl or other SHACL files.")

In [ ]:
def get_property_value(df, subject: str, predicate: str) -> Optional[str]:
    """Get single property value for a subject."""
    rows = df.filter(
        (F.col("subject") == subject) & (F.col("predicate") == predicate)
    ).select("object").collect()
    return rows[0].object if rows else None


def get_property_values(df, subject: str, predicate: str) -> List[str]:
    """Get all property values for a subject."""
    rows = df.filter(
        (F.col("subject") == subject) & (F.col("predicate") == predicate)
    ).select("object").collect()
    return [row.object for row in rows]


def parse_rdf_list(df, list_head: str) -> List[str]:
    """Parse an RDF list (rdf:first/rdf:rest chain) into Python list."""
    items = []
    current = list_head
    while current and current != RDF_NIL:
        first = get_property_value(df, current, RDF_FIRST)
        if first:
            items.append(first)
        current = get_property_value(df, current, RDF_REST)
    return items


def extract_local_name(uri: str) -> str:
    """Extract local name from URI."""
    if uri is None: return None
    if "#" in uri: return uri.split("#")[-1]
    if "/" in uri: return uri.split("/")[-1]
    return uri

In [ ]:
def parse_property_constraint(df, prop_shape_uri: str) -> PropertyConstraint:
    """Parse a sh:PropertyShape into a PropertyConstraint."""
    pc = PropertyConstraint()
    
    # sh:path
    pc.path = get_property_value(df, prop_shape_uri, SH_PATH)
    pc.path_name = extract_local_name(pc.path)
    
    # Datatype and class constraints
    pc.datatype = get_property_value(df, prop_shape_uri, SH_DATATYPE)
    pc.node_kind = get_property_value(df, prop_shape_uri, SH_NODE_KIND)
    pc.class_constraint = get_property_value(df, prop_shape_uri, SH_CLASS)
    
    # Cardinality
    min_count = get_property_value(df, prop_shape_uri, SH_MIN_COUNT)
    max_count = get_property_value(df, prop_shape_uri, SH_MAX_COUNT)
    pc.min_count = int(min_count) if min_count else None
    pc.max_count = int(max_count) if max_count else None
    
    # String constraints
    min_len = get_property_value(df, prop_shape_uri, SH_MIN_LENGTH)
    max_len = get_property_value(df, prop_shape_uri, SH_MAX_LENGTH)
    pc.min_length = int(min_len) if min_len else None
    pc.max_length = int(max_len) if max_len else None
    pc.pattern = get_property_value(df, prop_shape_uri, SH_PATTERN)
    
    # Value range
    pc.min_inclusive = get_property_value(df, prop_shape_uri, SH_MIN_INCLUSIVE)
    pc.max_inclusive = get_property_value(df, prop_shape_uri, SH_MAX_INCLUSIVE)
    pc.min_exclusive = get_property_value(df, prop_shape_uri, SH_MIN_EXCLUSIVE)
    pc.max_exclusive = get_property_value(df, prop_shape_uri, SH_MAX_EXCLUSIVE)
    
    # Enumerated values (sh:in)
    in_list = get_property_value(df, prop_shape_uri, SH_IN)
    if in_list:
        pc.allowed_values = parse_rdf_list(df, in_list)
    
    # Metadata
    pc.name = get_property_value(df, prop_shape_uri, SH_NAME)
    pc.description = get_property_value(df, prop_shape_uri, SH_DESCRIPTION)
    pc.message = get_property_value(df, prop_shape_uri, SH_MESSAGE)
    pc.severity = get_property_value(df, prop_shape_uri, SH_SEVERITY)
    
    return pc

In [ ]:
def parse_node_shape(df, shape_uri: str) -> NodeShape:
    """Parse a sh:NodeShape into a NodeShape object."""
    shape = NodeShape(uri=shape_uri)
    
    # Target (explicit)
    shape.target_class = get_property_value(df, shape_uri, SH_TARGET_CLASS)
    shape.target_node = get_property_value(df, shape_uri, SH_TARGET_NODE)
    
    # Implicit class targeting (SHACL 2.1.3): if shape URI is also a class, it targets itself
    if not shape.target_class:
        shape_types = get_property_values(df, shape_uri, RDF_TYPE)
        if OWL_CLASS in shape_types or RDFS_CLASS in shape_types:
            shape.target_class = shape_uri  # Implicit target
    
    # Closed shape
    closed = get_property_value(df, shape_uri, SH_CLOSED)
    shape.is_closed = closed == "true" if closed else False
    
    # Metadata
    shape.name = get_property_value(df, shape_uri, SH_NAME)
    shape.description = get_property_value(df, shape_uri, SH_DESCRIPTION)
    shape.message = get_property_value(df, shape_uri, SH_MESSAGE)
    shape.severity = get_property_value(df, shape_uri, SH_SEVERITY)
    
    # Property constraints
    prop_shape_uris = get_property_values(df, shape_uri, SH_PROPERTY)
    for prop_uri in prop_shape_uris:
        pc = parse_property_constraint(df, prop_uri)
        shape.property_constraints.append(pc)
    
    return shape

In [ ]:
# Find all NodeShapes
node_shape_uris = [
    row.subject for row in
    df_triples.filter(
        (F.col("predicate") == RDF_TYPE) &
        (F.col("object") == SH_NODE_SHAPE)
    ).select("subject").distinct().collect()
]

print(f"Found {len(node_shape_uris)} NodeShapes")

# Parse all NodeShapes
node_shapes: List[NodeShape] = []
for uri in node_shape_uris:
    shape = parse_node_shape(df_triples, uri)
    node_shapes.append(shape)
    print(f"  - {extract_local_name(uri)}: {len(shape.property_constraints)} property constraints")

In [ ]:
# Find standalone PropertyShapes (not nested in NodeShapes)
all_prop_shapes = set(
    row.subject for row in
    df_triples.filter(
        (F.col("predicate") == RDF_TYPE) &
        (F.col("object") == SH_PROPERTY_SHAPE)
    ).select("subject").distinct().collect()
)

# Property shapes referenced from NodeShapes
nested_prop_shapes = set(
    row.object for row in
    df_triples.filter(F.col("predicate") == SH_PROPERTY).select("object").collect()
)

standalone_prop_shapes = all_prop_shapes - nested_prop_shapes
print(f"\nStandalone PropertyShapes: {len(standalone_prop_shapes)}")

# Parse standalone property shapes
standalone_constraints: List[PropertyConstraint] = []
for uri in standalone_prop_shapes:
    pc = parse_property_constraint(df_triples, uri)
    standalone_constraints.append(pc)
    print(f"  - {pc.path_name}: {pc.datatype or pc.class_constraint or 'no type'}")

In [ ]:
# Summary statistics
print("\n" + "=" * 60)
print("SHACL SHAPE SUMMARY")
print("=" * 60)

total_prop_constraints = sum(len(s.property_constraints) for s in node_shapes)
total_prop_constraints += len(standalone_constraints)

# Count constraint types
constraint_types = {
    "minCount": 0, "maxCount": 0, "datatype": 0, "class": 0,
    "pattern": 0, "nodeKind": 0, "minInclusive": 0, "maxInclusive": 0,
    "minLength": 0, "maxLength": 0, "allowedValues": 0
}

def count_constraint(pc):
    if pc.min_count is not None: constraint_types["minCount"] += 1
    if pc.max_count is not None: constraint_types["maxCount"] += 1
    if pc.datatype: constraint_types["datatype"] += 1
    if pc.class_constraint: constraint_types["class"] += 1
    if pc.pattern: constraint_types["pattern"] += 1
    if pc.node_kind: constraint_types["nodeKind"] += 1
    if pc.min_inclusive: constraint_types["minInclusive"] += 1
    if pc.max_inclusive: constraint_types["maxInclusive"] += 1
    if pc.min_length is not None: constraint_types["minLength"] += 1
    if pc.max_length is not None: constraint_types["maxLength"] += 1
    if pc.allowed_values: constraint_types["allowedValues"] += 1

for shape in node_shapes:
    for pc in shape.property_constraints:
        count_constraint(pc)
for pc in standalone_constraints:
    count_constraint(pc)

print(f"\nNodeShapes: {len(node_shapes)}")
print(f"PropertyShapes (total): {total_prop_constraints}")
print(f"  - Nested in NodeShapes: {total_prop_constraints - len(standalone_constraints)}")
print(f"  - Standalone: {len(standalone_constraints)}")

print("\nConstraint Types Found:")
for ct, count in sorted(constraint_types.items(), key=lambda x: -x[1]):
    if count > 0:
        print(f"  {ct}: {count}")

In [ ]:
# Show shapes by target class
print("\n" + "=" * 60)
print("SHAPES BY TARGET CLASS")
print("=" * 60)

for shape in sorted(node_shapes, key=lambda s: s.name or s.uri):
    target = extract_local_name(shape.target_class) if shape.target_class else "(no target)"
    name = shape.name or extract_local_name(shape.uri)
    print(f"\n{name} -> {target}")
    print(f"  Closed: {shape.is_closed}")
    print(f"  Property constraints: {len(shape.property_constraints)}")
    
    for pc in shape.property_constraints[:5]:  # Show first 5
        constraints = []
        if pc.min_count is not None: constraints.append(f"min={pc.min_count}")
        if pc.max_count is not None: constraints.append(f"max={pc.max_count}")
        if pc.datatype: constraints.append(f"type={extract_local_name(pc.datatype)}")
        if pc.class_constraint: constraints.append(f"class={extract_local_name(pc.class_constraint)}")
        if pc.pattern: constraints.append(f"pattern={pc.pattern}")
        
        constraint_str = ", ".join(constraints) if constraints else "(no constraints)"
        print(f"    - {pc.path_name}: {constraint_str}")
    
    if len(shape.property_constraints) > 5:
        print(f"    ... and {len(shape.property_constraints) - 5} more")

In [ ]:
# Build JSON output for UI/frontend
shacl_output = {
    "nodeShapes": [s.to_dict() for s in node_shapes],
    "standaloneConstraints": [pc.to_dict() for pc in standalone_constraints],
    "summary": {
        "nodeShapeCount": len(node_shapes),
        "propertyConstraintCount": total_prop_constraints,
        "constraintTypes": {k: v for k, v in constraint_types.items() if v > 0}
    }
}

print("\n" + "=" * 60)
print("JSON OUTPUT (first 2000 chars)")
print("=" * 60)
json_output = json.dumps(shacl_output, indent=2)
print(json_output[:2000])
if len(json_output) > 2000:
    print(f"\n... ({len(json_output)} total characters)")

In [ ]:
# Save JSON to lakehouse
from datetime import datetime

shacl_output["generatedAt"] = datetime.now().isoformat()

with open(OUTPUT_JSON, 'w') as f:
    json.dump(shacl_output, f, indent=2)

print(f"\nSHACL shapes saved to: {OUTPUT_JSON}")

In [ ]:
# Write to Delta table for pipeline use
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType

# Explicit schema to handle None values
shape_schema = StructType([
    StructField("shape_uri", StringType(), True),
    StructField("shape_name", StringType(), True),
    StructField("target_class", StringType(), True),
    StructField("is_closed", BooleanType(), True),
    StructField("property_path", StringType(), True),
    StructField("property_name", StringType(), True),
    StructField("datatype", StringType(), True),
    StructField("node_kind", StringType(), True),
    StructField("class_constraint", StringType(), True),
    StructField("min_count", IntegerType(), True),
    StructField("max_count", IntegerType(), True),
    StructField("pattern", StringType(), True),
    StructField("min_inclusive", StringType(), True),
    StructField("max_inclusive", StringType(), True),
    StructField("constraint_json", StringType(), True),
])

# Flatten shapes into table rows
shape_rows = []
for shape in node_shapes:
    for pc in shape.property_constraints:
        shape_rows.append(Row(
            shape_uri=shape.uri,
            shape_name=shape.name or extract_local_name(shape.uri),
            target_class=shape.target_class,
            is_closed=shape.is_closed,
            property_path=pc.path,
            property_name=pc.path_name,
            datatype=pc.datatype,
            node_kind=pc.node_kind,
            class_constraint=pc.class_constraint,
            min_count=pc.min_count,
            max_count=pc.max_count,
            pattern=pc.pattern,
            min_inclusive=pc.min_inclusive,
            max_inclusive=pc.max_inclusive,
            constraint_json=json.dumps(pc.to_dict())
        ))

# Add standalone constraints
for pc in standalone_constraints:
    shape_rows.append(Row(
        shape_uri=None,
        shape_name="(standalone)",
        target_class=None,
        is_closed=False,
        property_path=pc.path,
        property_name=pc.path_name,
        datatype=pc.datatype,
        node_kind=pc.node_kind,
        class_constraint=pc.class_constraint,
        min_count=pc.min_count,
        max_count=pc.max_count,
        pattern=pc.pattern,
        min_inclusive=pc.min_inclusive,
        max_inclusive=pc.max_inclusive,
        constraint_json=json.dumps(pc.to_dict())
    ))

if shape_rows:
    df_shapes = spark.createDataFrame(shape_rows, schema=shape_schema)
    df_shapes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
    print(f"\nSaved {len(shape_rows)} constraints to '{OUTPUT_TABLE}'")
else:
    print("\nNo SHACL shapes to save.")


In [ ]:
print("\n" + "=" * 60)
print("F6.1 SHACL SHAPE PARSER COMPLETE")
print("=" * 60)
print(f"\nOutputs:")
print(f"  - Delta table: {OUTPUT_TABLE}")
print(f"  - JSON file: {OUTPUT_JSON}")
print(f"\nNodeShapes: {len(node_shapes)}")
print(f"Property constraints: {total_prop_constraints}")
print(f"\nNext: Run 11_shacl_validator (F6.2) to validate data against these shapes.")